# 標準住宅の外皮性能

床断熱・基礎断熱の等級4～7について、面積と層構成からUA・ηAC・ηAHを計算します。窓のηdは枠・ガラス・付属部材から算定でき、仕様が未入力の窓では警告とともに季節別の不利側を採用します。

In [ ]:
%pip -q install git+https://github.com/iguchi-lab/vtsimnx.git@d27eb468bd67f2e14c83be875bd1b48a942fb370

In [ ]:
import json
from pathlib import Path
import pandas as pd
import vtsimnx as vt

model = "foundation5"  # 床断熱は default, grade5～grade7。基礎断熱は foundation4～foundation7。
region = 6
house = vt.standard_house
types, layers = house.surface_types[model], house.layers[model]
input_data = {"surfaces": [
    {**row, "layers": layers.get(types[row["surface"]].get("layer_key"))}
    for row in house.surface_rows[model]
]}
# 室温計算用の input_data を組み立てた場合は、その辞書を直接使用できる。
path = Path(f"{model}_region{region}_rating.json")
if not path.exists():
    assessment = house.rating_template(model, region=region)
    path.write_text(json.dumps(assessment, ensure_ascii=False, indent=2), encoding="utf-8")
print(path.resolve())

JSONの窓に、例えば `frame_type` = 金属製など、`glass_type` = 一般複層、`attachment` = なし と入力するとηdは0.632です。既存JSONにもこの3項目を追加できます。仕様の空欄は不利側を採用し、部位別結果に採用値を表示します。直接入力したηdも使えます。窓のUは設計値、または詳細な枠・ガラス・中空層仕様から設定できます。ドアのUと土間外周のψには設計値を入力します。土間外周は実長 `length` [m] を優先します。省略時は対応する基礎立上りの面積 `wall_area` [m²] または立上り面のキー一覧 `wall_keys` を指定してください。警告を表示し、面積を仮定高さ0.4 mで割って長さを推定します。新規の基礎断熱モデルは外周35.49 mを入力済みです。保存済みJSONを使う場合は、新しいテンプレートと窓仕様・外周長を照合してください。[選択肢と計算方法](https://github.com/iguchi-lab/vtsimnx/blob/d27eb468bd67f2e14c83be875bd1b48a942fb370/docs/envelope_rating.md)

In [ ]:
from vtsimnx.window_rating import window_u

assessment = json.loads(path.read_text(encoding="utf-8"))
missing = []
for part in assessment["parts"]:
    name = part["keys"][0]
    if part["kind"] == "window" and part.get("U") is None:
        try:
            window_u(part)
        except ValueError as exc:
            missing.append((name, "U または窓仕様", str(exc)))
    if part["kind"] == "door" and part.get("U") is None:
        missing.append((name, "U", "開口部全体の熱貫流率 [W/(m²·K)]"))
for line in assessment["lines"]:
    perimeter = line.get("kind") == "perimeter"
    can_estimate = perimeter and (line.get("wall_area") is not None or bool(line.get("wall_keys")))
    if line.get("length") is None and not can_estimate:
        label = "実長 [m]、または立上りの wall_area [m²] / wall_keys" if perimeter else "長さ [m]"
        missing.append((line["name"], "length", label))
    for field, label in [("psi", "線熱貫流率 [W/(m·K)]"), ("H", "温度差係数")]:
        if line.get(field) is None:
            missing.append((line["name"], field, label))

if missing:
    print(f"{path.name} に以下の設計値を入力して保存し、このセルを再実行してください。")
    display(pd.DataFrame(missing, columns=["部位", "JSONの項目", "必要な設計値"]))
else:
    result = vt.envelope_performance(input_data, assessment)
    display(pd.DataFrame([{k: v for k, v in result.items() if k != "parts"}]))
    display(pd.DataFrame(result["parts"]))

### 室温計算にも同じ窓を使う入力例
金属枠・一般複層（中空層6 mm）を選びます。例の対流割合は25%と仮定しています。下のセルで評価用と室温計算用の入力を確認できます。

In [ ]:
w = vt.window_properties(frame_type="金属製", glass_type="一般複層", gap_mm=6,
                         attachment="なし", convective_fraction=0.25)
print("外皮評価", w["rating"])
print("室温計算の窓面", w["surface"])